In [1]:
!pip install spacy nltk pdfplumber docx2txt scikit-learn torch transformers faiss-cpu requests gradio
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----------------------- ---------------- 7.6/12.8 MB 39.0 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 35.8 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 35.8 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 35.8 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 35.8 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 35.8 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 9.0 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\GAD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [3]:
import os
import json
import pdfplumber
import docx2txt
import requests
import spacy
import gradio as gr
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from nltk.tokenize import word_tokenize

# Load NLP Models
nlp = spacy.load("en_core_web_sm")
bert_model = SentenceTransformer("all-MiniLM-L6-v2")  # Lightweight BERT for embeddings

# Ollama API for LLaMA 3.2
OLLAMA_API = "http://localhost:11434/api/chat"

# Predefined Job Descriptions
job_descriptions = {
    "Software Engineer": "Looking for a software engineer proficient in Python, Java, and cloud technologies. Experience in AI/ML is a plus.",
    "Data Scientist": "Seeking a data scientist with expertise in machine learning, Python, and data visualization using Tableau or PowerBI.",
    "Cyber Security Analyst": "Need a cyber security expert familiar with penetration testing, network security, and compliance frameworks like ISO 27001."
}

# Function to Extract Text from Resume (PDF/DOCX)
def extract_text_from_resume(file_path):
    text = ""
    try:
        if file_path.endswith(".pdf"):
            with pdfplumber.open(file_path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() + "\n"
        elif file_path.endswith(".docx"):
            text = docx2txt.process(file_path)
        return text.strip()
    except Exception as e:
        return f"Error extracting text: {e}"

# Function to Parse Resume (Extract Skills, Education, Experience, Projects)
def parse_resume(text):
    doc = nlp(text)
    skills, experience, education, projects = set(), set(), set(), set()
    
    for ent in doc.ents:
        if ent.label_ in ["ORG", "GPE"]:
            education.add(ent.text)
        elif ent.label_ in ["DATE", "TIME"]:
            experience.add(ent.text)
        elif ent.label_ == "PRODUCT":
            skills.add(ent.text)
    
    tokens = word_tokenize(text.lower())
    if "project" in tokens:
        projects.add(text[text.lower().find("project"):])

    return {
        "skills": list(skills),
        "experience": list(experience),
        "education": list(education),
        "projects": list(projects)
    }

# Function to Get LLaMA Response (Optional)
def chat_with_llama(prompt):
    data = {
        "model": "llama3.2",
        "messages": [{"role": "user", "content": prompt}]
    }
    try:
        response = requests.post(OLLAMA_API, json=data)
        return response.json().get("message", {}).get("content", "No response")
    except requests.exceptions.RequestException as e:
        return f"Error connecting to LLaMA: {e}"

# Function to Compute Matching Score (TF-IDF & BERT)
def compute_match_score(resume_text, job_text):
    try:
        # TF-IDF Matching
        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform([resume_text, job_text])
        tfidf_similarity = (tfidf_matrix * tfidf_matrix.T).toarray()[0, 1]
        
        # BERT Embeddings Similarity
        embeddings = bert_model.encode([resume_text, job_text])
        cosine_similarity = np.dot(embeddings[0], embeddings[1]) / (np.linalg.norm(embeddings[0]) * np.linalg.norm(embeddings[1]))

        # Final Matching Score (Weighted)
        final_score = (0.5 * tfidf_similarity) + (0.5 * cosine_similarity)
        return round(final_score * 100, 2)
    except Exception as e:
        return f"Error calculating match score: {e}"

def process_resume(file_path, selected_role):
    if not file_path:
        return "Please upload a resume file."
    
    resume_text = extract_text_from_resume(file_path)
    parsed_data = parse_resume(resume_text)
    
    job_text = job_descriptions.get(selected_role, "No job description available.")
    match_score = compute_match_score(resume_text, job_text)
    
    llama_feedback = chat_with_llama(f"Based on this resume: {resume_text}\nHow well does it match this job description: {job_text}?")

    return f"**Match Score:** {match_score}%\n\n**LLaMA Insights:** {llama_feedback}\n\n**Extracted Skills:** {parsed_data['skills']}\n\n**Experience:** {parsed_data['experience']}\n\n**Education:** {parsed_data['education']}\n\n**Projects:** {parsed_data['projects']}"

# Gradio UI Setup
gr.Interface(
    fn=process_resume,
    inputs=[gr.File(type="filepath"), gr.Dropdown(choices=["Software Engineer", "Data Scientist", "Cyber Security Analyst"])],
    outputs="text"
).launch()


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [4]:
import json
import re
import nltk
import spacy
import gradio as gr
import pdfplumber
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer

# Download necessary NLTK resources
nltk.download("punkt")
nltk.download("stopwords")

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Define job roles and required skills
JOB_ROLES = {
    "Data Scientist": {"Python", "SQL", "Machine Learning", "Deep Learning", "NLP", "Statistics"},
    "Software Engineer": {"Python", "Java", "C++", "Git", "OOP", "Algorithms"},
    "Cloud Engineer": {"AWS", "Azure", "Docker", "Kubernetes", "Terraform", "Networking"},
    "Cybersecurity Analyst": {"Cybersecurity", "Ethical Hacking", "Network Security", "Penetration Testing"},
    "AI Engineer": {"Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "AI"}
}

def extract_text_from_pdf(pdf_file):
    """Extract text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text if text else "Error extracting text from PDF"

def extract_skills(text):
    """Extract skills from resume using predefined list and NLP."""
    extracted_skills = set()
    text_lower = text.lower()

    # Match predefined skills
    for job, skills in JOB_ROLES.items():
        for skill in skills:
            if skill.lower() in text_lower:
                extracted_skills.add(skill)

    # Use spaCy for Named Entity Recognition (NER)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PRODUCT"]:  # Modify based on dataset
            extracted_skills.add(ent.text)

    return list(extracted_skills)

def calculate_match_score(resume_skills, job_role):
    """Calculate match score between extracted skills and job role requirements."""
    required_skills = JOB_ROLES.get(job_role, set())
    if not required_skills:
        return 0.0

    matched_skills = set(resume_skills) & required_skills
    match_percentage = (len(matched_skills) / len(required_skills)) * 100
    return round(match_percentage, 2)

def process_resume(file, job_role):
    """Process the uploaded resume, extract skills, and calculate match score."""
    # Extract text from PDF
    resume_text = extract_text_from_pdf(file.name)
    
    # Extract skills from resume
    extracted_skills = extract_skills(resume_text)
    
    # Compute match score
    match_score = calculate_match_score(extracted_skills, job_role)

    return f"🔹 **Match Score:** {match_score}%\n\n🔹 **Extracted Skills:** {', '.join(extracted_skills) if extracted_skills else 'No relevant skills found'}\n\n🔹 **Required Skills for {job_role}:** {', '.join(JOB_ROLES[job_role])}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 📄 Resume Skill Matcher")
    gr.Markdown("Upload your resume and select a job role to see your match score.")
    
    file_input = gr.File(label="Upload Resume (PDF)")
    job_dropdown = gr.Dropdown(choices=list(JOB_ROLES.keys()), label="Select Job Role")
    output_text = gr.Textbox(label="Result", interactive=False)
    
    submit_button = gr.Button("Check Match Score")
    submit_button.click(fn=process_resume, inputs=[file_input, job_dropdown], outputs=output_text)

# Run the Gradio app
demo.launch()


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\GAD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\GAD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [5]:
import json
import re
import nltk
import spacy
import gradio as gr
import pdfplumber
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Download necessary NLTK resources
nltk.download("punkt")
nltk.download("stopwords")

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Define job roles and required skills
JOB_ROLES = {
    "Data Scientist": {"Python", "SQL", "Machine Learning", "Deep Learning", "NLP", "Statistics"},
    "Software Engineer": {"Python", "Java", "C++", "Git", "OOP", "Algorithms"},
    "Cloud Engineer": {"AWS", "Azure", "Docker", "Kubernetes", "Terraform", "Networking"},
    "Cybersecurity Analyst": {"Cybersecurity", "Ethical Hacking", "Network Security", "Penetration Testing"},
    "AI Engineer": {"Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "AI"}
}

def extract_text_from_pdf(pdf_file):
    """Extract text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text if text else "Error extracting text from PDF"

def extract_skills(text):
    """Extract skills from resume using predefined list and NLP."""
    extracted_skills = set()
    text_lower = text.lower()

    # Match predefined skills
    for job, skills in JOB_ROLES.items():
        for skill in skills:
            if skill.lower() in text_lower:
                extracted_skills.add(skill)

    # Use spaCy for Named Entity Recognition (NER)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PRODUCT"]:  # Modify based on dataset
            extracted_skills.add(ent.text)

    return list(extracted_skills)

def calculate_match_score(resume_skills, job_role):
    """Calculate match score and identify matched/missing skills."""
    required_skills = JOB_ROLES.get(job_role, set())
    if not required_skills:
        return 0.0, set(), set()

    matched_skills = set(resume_skills) & required_skills
    missing_skills = required_skills - matched_skills
    match_percentage = (len(matched_skills) / len(required_skills)) * 100

    return round(match_percentage, 2), matched_skills, missing_skills

def process_resume(file, job_role):
    """Process resume, extract skills, and calculate match score."""
    resume_text = extract_text_from_pdf(file.name)
    extracted_skills = extract_skills(resume_text)
    match_score, matched_skills, missing_skills = calculate_match_score(extracted_skills, job_role)

    return f"🔹 **Match Score:** {match_score}%\n\n" \
           f"🔹 **Extracted Skills:** {', '.join(extracted_skills) if extracted_skills else 'None'}\n\n" \
           f"🔹 **Matched Skills:** {', '.join(matched_skills) if matched_skills else 'None'}\n\n" \
           f"🔹 **Missing Skills:** {', '.join(missing_skills) if missing_skills else 'None'}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 📄 Resume Skill Matcher")
    gr.Markdown("Upload your resume and select a job role to see your match score.")

    file_input = gr.File(label="Upload Resume (PDF)")
    job_dropdown = gr.Dropdown(choices=list(JOB_ROLES.keys()), label="Select Job Role")
    output_text = gr.Textbox(label="Result", interactive=False)

    submit_button = gr.Button("Check Match Score")
    submit_button.click(fn=process_resume, inputs=[file_input, job_dropdown], outputs=output_text)

# Run the Gradio app
demo.launch(share = True)


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\GAD\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\GAD\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://639d94ba07a0470756.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [6]:
# Define job roles and required skills
JOB_ROLES = {
    "Data Scientist": {"Python", "SQL", "Machine Learning", "Deep Learning", "NLP", "Statistics", "Pandas", "Scikit-Learn"},
    "Software Engineer": {"Python", "Java", "C++", "Git", "OOP", "Algorithms"},
    "Cloud Engineer": {"AWS", "Azure", "Docker", "Kubernetes", "Terraform", "Networking"},
    "Cybersecurity Analyst": {"Cybersecurity", "Ethical Hacking", "Network Security", "Penetration Testing"},
    "AI Engineer": {"Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "AI"}
}

# Predefined list of common skills (to avoid extracting unnecessary words)
COMMON_SKILLS = set([
    "Python", "Java", "C++", "SQL", "Machine Learning", "Deep Learning", "NLP", "Pandas", "Scikit-Learn",
    "TensorFlow", "PyTorch", "Data Analysis", "Cybersecurity", "Ethical Hacking", "AWS", "Azure", "Docker",
    "Kubernetes", "Flask", "Django", "Linux", "JavaScript", "React", "Node.js", "Computer Vision", "Statistics",
    "Mathematics", "Tableau", "Power BI", "Time Management", "Problem Solving", "Communication", "Teamwork"
])

def extract_text_from_pdf(pdf_file):
    """Extract text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text if text else "Error extracting text from PDF"

def extract_skills(text):
    """Extract skills from resume using predefined list and NLP."""
    extracted_skills = set()
    text_lower = text.lower()

    # Match predefined skills
    for skill in COMMON_SKILLS:
        if skill.lower() in text_lower:
            extracted_skills.add(skill)

    # Use spaCy for Named Entity Recognition (NER)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PERSON", "GPE", "FACILITY", "EVENT"]:  # Avoid extracting non-skills
            continue
        if ent.text in COMMON_SKILLS:  # Extract only valid skills
            extracted_skills.add(ent.text)

    return list(extracted_skills)

def calculate_match_score(resume_skills, job_role):
    """Calculate match score and identify matched/missing skills."""
    required_skills = JOB_ROLES.get(job_role, set())
    if not required_skills:
        return 0.0, set(), set()

    matched_skills = set(resume_skills) & required_skills
    missing_skills = required_skills - matched_skills
    match_percentage = (len(matched_skills) / len(required_skills)) * 100

    return round(match_percentage, 2), matched_skills, missing_skills

def process_resume(file, job_role):
    """Process resume, extract skills, and calculate match score."""
    resume_text = extract_text_from_pdf(file.name)
    extracted_skills = extract_skills(resume_text)
    match_score, matched_skills, missing_skills = calculate_match_score(extracted_skills, job_role)

    return f"🔹 **Match Score:** {match_score}%\n\n" \
           f"🔹 **Extracted Skills:** {', '.join(extracted_skills) if extracted_skills else 'None'}\n\n" \
           f"🔹 **Matched Skills:** {', '.join(matched_skills) if matched_skills else 'None'}\n\n" \
           f"🔹 **Missing Skills:** {', '.join(missing_skills) if missing_skills else 'None'}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 📄 Resume Skill Matcher")
    gr.Markdown("Upload your resume and select a job role to see your match score.")

    file_input = gr.File(label="Upload Resume (PDF)")
    job_dropdown = gr.Dropdown(choices=list(JOB_ROLES.keys()), label="Select Job Role")
    output_text = gr.Textbox(label="Result", interactive=False)

    submit_button = gr.Button("Check Match Score")
    submit_button.click(fn=process_resume, inputs=[file_input, job_dropdown], outputs=output_text)

# Run the Gradio app
demo.launch(share = True)


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://dd956ebbb26254dc49.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
# Define job roles and required skills
JOB_ROLES = {
    "Data Scientist": {"Python", "SQL", "Machine Learning", "Deep Learning", "NLP", "Statistics", "Pandas", "Scikit-Learn"},
    "Software Engineer": {"Python", "Java", "C++", "Git", "OOP", "Algorithms"},
    "Cloud Engineer": {"AWS", "Azure", "Docker", "Kubernetes", "Terraform", "Networking"},
    "Cybersecurity Analyst": {"Cybersecurity", "Ethical Hacking", "Network Security", "Penetration Testing"},
    "AI Engineer": {"Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "AI"}
}

# Predefined common skills
COMMON_SKILLS = {
    "Python", "Java", "C++", "SQL", "Machine Learning", "Deep Learning", "NLP", "Pandas", "Scikit-Learn",
    "TensorFlow", "PyTorch", "Data Analysis", "Cybersecurity", "Ethical Hacking", "AWS", "Azure", "Docker",
    "Kubernetes", "Flask", "Django", "Linux", "JavaScript", "React", "Node.js", "Computer Vision", "Statistics",
    "Mathematics", "Tableau", "Power BI", "Time Management", "Problem Solving", "Communication", "Teamwork"
}

# Learning recommendations for missing skills
LEARNING_RESOURCES = {
    "Python": "https://www.udemy.com/course/python-for-data-science-and-machine-learning-bootcamp/",
    "SQL": "https://www.coursera.org/learn/sql-for-data-science",
    "Machine Learning": "https://www.coursera.org/learn/machine-learning",
    "Deep Learning": "https://www.udemy.com/course/deep-learning-a-z/",
    "NLP": "https://www.udemy.com/course/nlp-natural-language-processing-with-python/",
    "Statistics": "https://www.khanacademy.org/math/statistics-probability",
    "AWS": "https://www.udemy.com/course/aws-certified-solutions-architect-associate/",
    "Cybersecurity": "https://www.udemy.com/course/the-complete-cyber-security-course-hacker-exposed/",
}

def extract_text_from_pdf(pdf_file):
    """Extract text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text if text else "Error extracting text from PDF"

def extract_skills(text):
    """Extract skills from resume using predefined list and NLP."""
    extracted_skills = set()
    text_lower = text.lower()

    # Match predefined skills
    for skill in COMMON_SKILLS:
        if skill.lower() in text_lower:
            extracted_skills.add(skill)

    # Use spaCy for Named Entity Recognition (NER)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PERSON", "GPE", "FACILITY", "EVENT"]:  # Avoid extracting non-skills
            continue
        if ent.text in COMMON_SKILLS:  # Extract only valid skills
            extracted_skills.add(ent.text)

    return list(extracted_skills)

def extract_summary(text):
    """Extract a brief summary from resume using NLP."""
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents][:3]  # Extract first 3 sentences
    return " ".join(sentences) if sentences else "Summary not found."

def calculate_match_score(resume_skills, job_role):
    """Calculate match score and identify matched/missing skills."""
    required_skills = JOB_ROLES.get(job_role, set())
    if not required_skills:
        return 0.0, set(), set(), []

    matched_skills = set(resume_skills) & required_skills
    missing_skills = required_skills - matched_skills
    match_percentage = (len(matched_skills) / len(required_skills)) * 100

    # Generate learning recommendations
    learning_links = [f"{skill}: {LEARNING_RESOURCES.get(skill, 'No course available')}" for skill in missing_skills]

    return round(match_percentage, 2), matched_skills, missing_skills, learning_links

def process_resume(file, job_role):
    """Process resume, extract skills, summary, and calculate match score."""
    resume_text = extract_text_from_pdf(file.name)
    extracted_skills = extract_skills(resume_text)
    summary = extract_summary(resume_text)
    match_score, matched_skills, missing_skills, learning_links = calculate_match_score(extracted_skills, job_role)

    return f"🔹 **Match Score:** {match_score}%\n\n" \
           f"🔹 **Resume Summary:** {summary}\n\n" \
           f"🔹 **Extracted Skills:** {', '.join(extracted_skills) if extracted_skills else 'None'}\n\n" \
           f"🔹 **Matched Skills:** {', '.join(matched_skills) if matched_skills else 'None'}\n\n" \
           f"🔹 **Missing Skills:** {', '.join(missing_skills) if missing_skills else 'None'}\n\n" \
           f"🔹 **Learning Recommendations:**\n{chr(10).join(learning_links) if learning_links else 'No recommendations'}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 📄 Resume Skill Matcher & Learning Recommendations")
    gr.Markdown("Upload your resume and select a job role to check your match score and get learning recommendations.")

    file_input = gr.File(label="Upload Resume (PDF)")
    job_dropdown = gr.Dropdown(choices=list(JOB_ROLES.keys()), label="Select Job Role")
    output_text = gr.Textbox(label="Result", interactive=False)

    submit_button = gr.Button("Check Match Score")
    submit_button.click(fn=process_resume, inputs=[file_input, job_dropdown], outputs=output_text)

# Run the Gradio app
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* Running on public URL: https://9a1a3eb288d21c0ce1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [8]:
# Define job roles and required skills
JOB_ROLES = {
    "Data Scientist": {"Python", "SQL", "Machine Learning", "Deep Learning", "NLP", "Statistics", "Pandas", "Scikit-Learn"},
    "Software Engineer": {"Python", "Java", "C++", "Git", "OOP", "Algorithms"},
    "Cloud Engineer": {"AWS", "Azure", "Docker", "Kubernetes", "Terraform", "Networking"},
    "Cybersecurity Analyst": {"Cybersecurity", "Ethical Hacking", "Network Security", "Penetration Testing"},
    "AI Engineer": {"Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "AI"}
}

# Predefined common skills
COMMON_SKILLS = {
    "Python", "Java", "C++", "SQL", "Machine Learning", "Deep Learning", "NLP", "Pandas", "Scikit-Learn",
    "TensorFlow", "PyTorch", "Data Analysis", "Cybersecurity", "Ethical Hacking", "AWS", "Azure", "Docker",
    "Kubernetes", "Flask", "Django", "Linux", "JavaScript", "React", "Node.js", "Computer Vision", "Statistics",
    "Mathematics", "Tableau", "Power BI", "Time Management", "Problem Solving", "Communication", "Teamwork"
}

# Learning recommendations for missing skills
LEARNING_RESOURCES = {
    "Python": "https://www.udemy.com/course/python-for-data-science-and-machine-learning-bootcamp/",
    "SQL": "https://www.coursera.org/learn/sql-for-data-science",
    "Machine Learning": "https://www.coursera.org/learn/machine-learning",
    "Deep Learning": "https://www.udemy.com/course/deep-learning-a-z/",
    "NLP": "https://www.udemy.com/course/nlp-natural-language-processing-with-python/",
    "Statistics": "https://www.khanacademy.org/math/statistics-probability",
    "AWS": "https://www.udemy.com/course/aws-certified-solutions-architect-associate/",
    "Cybersecurity": "https://www.udemy.com/course/the-complete-cyber-security-course-hacker-exposed/",
}

def extract_text_from_pdf(pdf_file):
    """Extract text from a PDF file."""
    text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
    return text if text else "Error extracting text from PDF"

def extract_skills(text):
    """Extract skills from resume using predefined list and NLP."""
    extracted_skills = set()
    text_lower = text.lower()

    # Match predefined skills
    for skill in COMMON_SKILLS:
        if skill.lower() in text_lower:
            extracted_skills.add(skill)

    # Use spaCy for Named Entity Recognition (NER)
    doc = nlp(text)
    for ent in doc.ents:
        if ent.label_ in ["ORG", "PERSON", "GPE", "FACILITY", "EVENT"]:  # Avoid extracting non-skills
            continue
        if ent.text in COMMON_SKILLS:  # Extract only valid skills
            extracted_skills.add(ent.text)

    return list(extracted_skills)

def extract_summary(text):
    """Extract a concise summary from the resume using NLP."""
    doc = nlp(text)
    for sent in doc.sents:
        if len(sent.text.split()) > 5:  # Ensure it's a meaningful sentence
            return sent.text
    return "Summary not found."

def calculate_match_score(resume_skills, job_role):
    """Calculate match score and identify matched/missing skills."""
    required_skills = JOB_ROLES.get(job_role, set())
    if not required_skills:
        return 0.0, set(), set(), []

    matched_skills = set(resume_skills) & required_skills
    missing_skills = required_skills - matched_skills
    match_percentage = (len(matched_skills) / len(required_skills)) * 100

    # Generate learning recommendations
    learning_links = [f"{skill}: {LEARNING_RESOURCES.get(skill, 'No course available')}" for skill in missing_skills]

    return round(match_percentage, 2), matched_skills, missing_skills, learning_links

def process_resume(file, job_role):
    """Process resume, extract skills, summary, and calculate match score."""
    resume_text = extract_text_from_pdf(file.name)
    extracted_skills = extract_skills(resume_text)
    summary = extract_summary(resume_text)
    match_score, matched_skills, missing_skills, learning_links = calculate_match_score(extracted_skills, job_role)

    return f"🔹 **Match Score:** {match_score}%\n\n" \
           f"🔹 **Resume Summary:** {summary}\n\n" \
           f"🔹 **Extracted Skills:** {', '.join(extracted_skills) if extracted_skills else 'None'}\n\n" \
           f"🔹 **Matched Skills:** {', '.join(matched_skills) if matched_skills else 'None'}\n\n" \
           f"🔹 **Missing Skills:** {', '.join(missing_skills) if missing_skills else 'None'}\n\n" \
           f"🔹 **Learning Recommendations:**\n{chr(10).join(learning_links) if learning_links else 'No recommendations'}"

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 📄 Resume Skill Matcher & Learning Recommendations")
    gr.Markdown("Upload your resume and select a job role to check your match score and get learning recommendations.")

    file_input = gr.File(label="Upload Resume (PDF)")
    job_dropdown = gr.Dropdown(choices=list(JOB_ROLES.keys()), label="Select Job Role")
    output_text = gr.Textbox(label="Result", interactive=False)

    submit_button = gr.Button("Check Match Score")
    submit_button.click(fn=process_resume, inputs=[file_input, job_dropdown], outputs=output_text)

# Run the Gradio app
demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://55a2f6673f216e0085.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
